<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Traces one student's records across prepared parquet artifacts to debug timeline or feature inconsistencies.

**Notebook Shape:** 24 cells (11 code, 13 markdown).

**Inputs / Data Sources:**
- `return pd.read_parquet(path, columns=columns)`

**Outputs / Side Effects:**
- `No explicit persisted output detected; side effects are limited to notebook display state unless cells are edited.`

**Logic Flow:**
1. Define parquet-loading helpers.
2. Load selected artifacts with optional columns.
3. Filter records for the target student.
4. Compare how the student appears across pipeline stages.

**Maintainability Notes:** Useful for debugging, but target IDs and paths should be parameterized before using it as a repeatable diagnostic tool.


# Debug: trace one `student_id` through the whole pipeline

**Why this notebook exists.** A `student_id` appears in `data/final/without_outliers.parquet`
but is reported missing from the cleaned source tables. This notebook finds *where* in the
file lineage the student appears/disappears, and flags **stale artifacts**.

**Decisive fact about the merge** (`mergecrgadd.ipynb`): `df_crg` is the LEFT/base table:
`df_crg.merge(df_add_snapshot, how="left").merge(df_acd, how="left")`. So the student set of
the final file == the CRG-clean student set *at the time the merge was last run*. A student in
the final but not in current CRG-clean therefore means the final was built from an **older**
CRG-clean (stale downstream artifacts), OR the membership check used the wrong dtype.

Set `TARGET_STUDENT_ID` and run top to bottom.

In [ ]:
import os
import pandas as pd

from src.paths import DATA_DIR

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# ---- configure here -------------------------------------------------
TARGET_STUDENT_ID = "10428.111"   # matched robustly as both string and float
ROOT = DATA_DIR
# ---------------------------------------------------------------------

# Pipeline artifacts in dependency order (upstream -> downstream).
PIPELINE = [
    ("raw_crg",   os.path.join(ROOT, r"raw\v_crg_student_course_raw.parquet")),
    ("raw_add",   os.path.join(ROOT, r"raw\v_add_student_degree_status.parquet")),
    ("raw_acd",   os.path.join(ROOT, r"raw\v_acd_degree_course.parquet")),
    ("clean_crg", os.path.join(ROOT, r"preprocessed\V_CRG_STUDENT_COURSE\clean_v_crg_student_course.parquet")),
    ("clean_add_SUBFOLDER (merge reads this)",
        os.path.join(ROOT, r"preprocessed\V_ADD_STUDENT_DEGREE_STATUS\clean_v_add_student_degree_status.parquet")),
    ("clean_add_ROOT (notebook writes this, PRE-drop)",
        os.path.join(ROOT, r"archive\pre_7c\v_add_student_degree_status_clean.parquet")),
    ("clean_acd", os.path.join(ROOT, r"preprocessed\V_ACD_DEGREE_COURSE\clean_v_acd_degree_course.parquet")),
    ("merge_crg_add_acd", os.path.join(ROOT, r"preprocessed\merge\merged_add_acd_crg.parquet")),
    ("merge_with_diploma", os.path.join(ROOT, r"preprocessed\merge\merged_with_diploma.parquet")),
    ("features_selected", os.path.join(ROOT, r"features\selected_model_population.parquet")),
    ("feature_engineered", os.path.join(ROOT, r"features\feature_engineered_primary.parquet")),
    ("final_without_outliers", os.path.join(ROOT, r"final\without_outliers.parquet")),
]

def student_mask(series, target=TARGET_STUDENT_ID):
    """Robust membership: match on string form AND numeric form."""
    target = str(target).strip()
    s_str = series.astype("string").str.strip()
    mask = (s_str == target)
    try:
        tnum = float(target)
        snum = pd.to_numeric(series, errors="coerce")
        mask = mask | (snum == tnum)
    except ValueError:
        pass
    return mask.fillna(False)

def load(path, columns=None):
    return pd.read_parquet(path, columns=columns)

print("Target student:", TARGET_STUDENT_ID)

## Stage 1 - Presence + freshness of the student in every artifact

For each file: does it exist, when was it last written (`mtime`), how many rows, the `student_id` dtype, and how many rows match the target (string and numeric).

In [37]:
rows = []
loaded = {}  # cache frames we will reuse later
for name, path in PIPELINE:
    rec = {"artifact": name, "exists": os.path.exists(path)}
    if not rec["exists"]:
        rec.update(mtime=None, rows=None, sid_dtype=None, str_hits=None, num_hits=None)
        rows.append(rec); continue
    rec["mtime"] = pd.Timestamp(os.path.getmtime(path), unit="s").round("s")
    try:
        df = load(path)
    except Exception as e:
        rec.update(rows=f"READ ERROR: {e}", sid_dtype=None, str_hits=None, num_hits=None)
        rows.append(rec); continue
    loaded[name] = df
    rec["rows"] = len(df)
    if "student_id" in df.columns:
        sid = df["student_id"]
        rec["sid_dtype"] = str(sid.dtype)
        s_str = sid.astype("string").str.strip()
        rec["str_hits"] = int((s_str == str(TARGET_STUDENT_ID)).sum())
        try:
            rec["num_hits"] = int((pd.to_numeric(sid, errors="coerce") == float(TARGET_STUDENT_ID)).sum())
        except ValueError:
            rec["num_hits"] = None
    else:
        rec.update(sid_dtype="<no student_id col>", str_hits=None, num_hits=None)
    rows.append(rec)

presence = pd.DataFrame(rows, columns=["artifact","exists","mtime","rows","sid_dtype","str_hits","num_hits"])
display(presence)

print("\nRead this table as:")
print(" - str_hits=0 but num_hits>0  -> membership check failed only because of dtype (string vs float).")
print(" - present downstream but 0 in clean_crg -> stale downstream (see Stage 2), or CRG rebuilt after merge.")

,artifact,exists,mtime,rows,sid_dtype,str_hits,num_hits
0,raw_crg,True,2026-05-31 11:21:06,1004459,float64,24.0,24.0
1,raw_add,True,2026-06-03 09:13:10,184530,float64,2.0,2.0
2,raw_acd,True,2026-06-01 10:39:45,4006,<no student_id col>,NaN,NaN
3,clean_crg,True,2026-06-20 08:45:45,761347,string,1.0,1.0
4,clean_add_SUBFOLDER (merge reads this),True,2026-06-20 08:48:02,179844,string,2.0,2.0
5,"clean_add_ROOT (notebook writes this, PRE-drop)",True,2026-06-20 08:48:01,181552,string,2.0,2.0
6,clean_acd,True,2026-06-17 09:10:08,4006,<no student_id col>,NaN,NaN
7,features_merged,True,2026-06-20 09:21:12,761347,string,1.0,1.0
8,audit_selected,True,2026-06-20 09:24:16,761346,string,1.0,1.0
9,final_without_outliers,True,2026-06-20 09:27:07,727852,string,1.0,1.0



Read this table as:
 - str_hits=0 but num_hits>0  -> membership check failed only because of dtype (string vs float).
 - present downstream but 0 in clean_crg -> stale downstream (see Stage 2), or CRG rebuilt after merge.


## Stage 2 - Staleness detection (mtime ordering)

Each downstream artifact must be **newer** than every upstream artifact it depends on. Any downstream file older than an input is stale and must be rebuilt.

In [38]:
DEPENDS_ON = {
    "clean_crg":               ["raw_crg"],
    "clean_add_SUBFOLDER (merge reads this)": ["raw_add"],
    "clean_acd":               ["raw_acd"],
    "features_merged":         ["clean_crg", "clean_add_SUBFOLDER (merge reads this)", "clean_acd"],
    "audit_selected":          ["features_merged"],
    "final_without_outliers":  ["audit_selected"],
}
mt = {name: (os.path.getmtime(path) if os.path.exists(path) else None)
      for name, path in PIPELINE}

stale_rows = []
for child, parents in DEPENDS_ON.items():
    c = mt.get(child)
    for p in parents:
        pp = mt.get(p)
        if c is None or pp is None:
            verdict = "MISSING FILE"
        elif pp > c:
            verdict = "STALE: parent newer than child"
        else:
            verdict = "ok"
        stale_rows.append({
            "child": child, "parent": p,
            "child_mtime": pd.Timestamp(c, unit="s").round("s") if c else None,
            "parent_mtime": pd.Timestamp(pp, unit="s").round("s") if pp else None,
            "verdict": verdict,
        })
staleness = pd.DataFrame(stale_rows)
display(staleness)

bad = staleness[staleness["verdict"].str.startswith(("STALE","MISSING"))]
print("FOUND STALE/MISSING LINKS -> rebuild those children:" if len(bad) else "All artifacts are fresh.")
display(bad)

,child,parent,child_mtime,parent_mtime,verdict
0,clean_crg,raw_crg,2026-06-20 08:45:45,2026-05-31 11:21:06,ok
1,clean_add_SUBFOLDER (merge reads this),raw_add,2026-06-20 08:48:02,2026-06-03 09:13:10,ok
2,clean_acd,raw_acd,2026-06-17 09:10:08,2026-06-01 10:39:45,ok
3,features_merged,clean_crg,2026-06-20 09:21:12,2026-06-20 08:45:45,ok
4,features_merged,clean_add_SUBFOLDER (merge reads this),2026-06-20 09:21:12,2026-06-20 08:48:02,ok
5,features_merged,clean_acd,2026-06-20 09:21:12,2026-06-17 09:10:08,ok
6,audit_selected,features_merged,2026-06-20 09:24:16,2026-06-20 09:21:12,ok
7,final_without_outliers,audit_selected,2026-06-20 09:27:07,2026-06-20 09:24:16,ok


All artifacts are fresh.


,child,parent,child_mtime,parent_mtime,verdict


## Stage 3 - Set diff: final vs CRG-clean

Because CRG is the left base of the merge, every final student must exist in the CRG-clean used at merge time. Students in `final` but not in current `clean_crg` are the smoking gun for a stale final (or a CRG file rebuilt after the merge).

In [39]:
if "final_without_outliers" in loaded and "clean_crg" in loaded:
    final_ids = set(loaded["final_without_outliers"]["student_id"].astype("string").str.strip())
    crg_ids   = set(loaded["clean_crg"]["student_id"].astype("string").str.strip())

    in_final_not_crg = final_ids - crg_ids
    print("students in FINAL:", len(final_ids))
    print("students in CRG-clean:", len(crg_ids))
    print("in FINAL but NOT in CRG-clean:", len(in_final_not_crg))
    tgt = str(TARGET_STUDENT_ID)
    print(f"\nIs target {tgt} in this 'final-only' set?  ->", tgt in in_final_not_crg)
    print("sample of final-only students:", list(sorted(in_final_not_crg))[:20])
else:
    print("Need both final_without_outliers and clean_crg loaded (check Stage 1).")

students in FINAL: 16141
students in CRG-clean: 16171
in FINAL but NOT in CRG-clean: 0

Is target 10428.111 in this 'final-only' set?  -> False
sample of final-only students: []


## Stage 4 - Re-derive: would the target survive CRG cleaning *today*?

Replays the CRG cleaning filters (from `clean_v_crg_student_course.ipynb`) on the **raw** CRG, restricted to the target student, and shows how many of the student's rows survive each step. If the student is dropped here by current rules but still sits in the final file, the final is stale relative to the current cleaning logic.

In [40]:
if "raw_crg" in loaded:
    raw = loaded["raw_crg"].copy()
    # normalize column names like the cleaning notebook does
    raw.columns = (pd.Index(raw.columns).astype("string").str.strip()
                   .str.replace(r"\s+", "_", regex=True)
                   .str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True)
                   .str.strip("_").str.lower())
    t = raw[student_mask(raw["student_id"])].copy()
    print(f"raw CRG rows for target: {len(t)}")

    def up(s):
        return s.astype("string").str.strip().str.upper()

    steps = []
    cur = t
    steps.append(("raw rows for student", len(cur)))

    if "active" in cur:
        cur = cur[up(cur["active"]).eq("A")]
        steps.append(("active == 'A'", len(cur)))
    if "register_status" in cur:
        cur = cur[up(cur["register_status"]).isin({"R", "E"})]
        steps.append(("register_status in {R,E}", len(cur)))
    if "finish_status" in cur:
        cur = cur[~up(cur["finish_status"]).eq("W")]
        steps.append(("drop finish_status == 'W'", len(cur)))
        cur = cur[up(cur["finish_status"]).isin({"P", "F", "FE", "FA"})]
        steps.append(("finish_status in {P,F,FE,FA}", len(cur)))
    crit = [c for c in ["student_course_id", "student_id", "course_id", "part_id", "course_credits"] if c in cur]
    cur = cur[~cur[crit].isna().any(axis=1)]
    steps.append(("non-null critical cols", len(cur)))
    if "course_credits" in cur:
        cc = pd.to_numeric(cur["course_credits"], errors="coerce")
        cur = cur[cc.gt(0)]
        steps.append(("course_credits > 0", len(cur)))
    if "final_mark" in cur:
        fm = pd.to_numeric(cur["final_mark"], errors="coerce")
        cur = cur[fm.isna() | fm.between(0, 100)]
        steps.append(("final_mark in [0,100] or null", len(cur)))

    display(pd.DataFrame(steps, columns=["step", "rows_surviving_for_target"]))
    print("\nIf rows_surviving reaches 0 at some step, that filter is why the student is NOT in")
    print("current CRG-clean. If >0, the student SHOULD be in CRG-clean -> rebuild CRG-clean (save is commented out!).")
    if "finish_status" in t:
        print("\ntarget finish_status distribution (raw):")
        display(up(t["finish_status"]).value_counts(dropna=False))
else:
    print("raw_crg not loaded.")

raw CRG rows for target: 24


,step,rows_surviving_for_target
0,raw rows for student,24
1,active == 'A',24
2,"register_status in {R,E}",5
3,drop finish_status == 'W',5
4,"finish_status in {P,F,FE,FA}",1
5,non-null critical cols,1
6,course_credits > 0,1
7,"final_mark in [0,100] or null",1



If rows_surviving reaches 0 at some step, that filter is why the student is NOT in
current CRG-clean. If >0, the student SHOULD be in CRG-clean -> rebuild CRG-clean (save is commented out!).

target finish_status distribution (raw):


finish_status
T    19
Z     4
F     1
Name: count, dtype: int64[pyarrow]

## Stage 5 - Raw ADD presence (secondary)

ADD is a left-merge, so it cannot *add* students to the final. This is only to understand the snapshot side. Note the two ADD-clean paths: the notebook's active save writes the **root** file *before* the only-in-ADD drop, while the merge reads the **subfolder** file (save commented out).

In [41]:
for key in ["raw_add",
            "clean_add_SUBFOLDER (merge reads this)",
            "clean_add_ROOT (notebook writes this, PRE-drop)"]:
    if key in loaded:
        df = loaded[key]
        hits = int(student_mask(df["student_id"]).sum()) if "student_id" in df else "n/a"
        print(f"{key:50} rows={len(df):>7}  target_rows={hits}")
    else:
        print(f"{key:50} NOT LOADED / MISSING")

raw_add                                            rows= 184530  target_rows=2
clean_add_SUBFOLDER (merge reads this)             rows= 179844  target_rows=2
clean_add_ROOT (notebook writes this, PRE-drop)    rows= 181552  target_rows=2


## How to interpret + fix

**Most likely root cause (from the code):** stale downstream artifacts. The two files the merge
consumes are *not written by the current notebooks*:

- CRG final save is **commented out** (`clean_v_crg_student_course.ipynb`, last cell).
- ADD writes the **root** pre-drop file (`v_add_student_degree_status_clean.parquet`) while the
  **subfolder** file the merge reads
  (`V_ADD_STUDENT_DEGREE_STATUS/clean_v_add_student_degree_status.parquet`) has its save **commented out**.

So edits to cleaning logic never reach the merge, and the final keeps old students.

**Decision guide from the stages above:**
- Stage 1 `str_hits=0, num_hits>0` everywhere -> it was only a dtype check problem; nothing is wrong.
- Stage 2 shows STALE links -> rebuild those children top-down.
- Stage 3 target in *final-only* set + Stage 4 student survives CRG filters -> CRG-clean on disk is
  stale: **uncomment the CRG save cell, re-run CRG cleaning, then re-run merge -> select -> outliers.**
- Stage 4 student is dropped at a filter -> current rules legitimately remove them; the final is just
  old. Rebuild the whole chain.

**Permanent fix:** uncomment the two save cells, make the ADD notebook save the post-drop frame to the
*subfolder* path the merge actually reads, and re-run the pipeline end to end so every artifact's mtime
is newer than its inputs.

## Stage 6 - Full FINAL row(s) for the target (transposed)

The final parquet is `df_model_a = df_primary.copy()`, so it still carries every diagnostic
column (raw inputs, fill source, flags). Dump the whole row(s) for the student so nothing is hidden.

In [42]:
fin = loaded.get("final_without_outliers")
if fin is None:
    print("final not loaded; run Stage 1 first.")
else:
    tgt_rows = fin[student_mask(fin["student_id"])].copy()
    print(f"final rows for target: {len(tgt_rows)}")
    with pd.option_context("display.max_rows", 400):
        display(tgt_rows.T)

final rows for target: 1


,26897
student_course_id,920764.111
student_id,10428.111
course_id,119.111
part_id,20151
degree_id,11.111
faculty_id,7.111
grade_id,684.111
final_mark,20
course_credits,2.0
attempt_number,1


## Stage 7 - WHY is `prev_gpa_points_clean` that value? (replay the exact fallback)

`prev_gpa_points_clean` is filled in priority order: **raw `prev_gpa_points` > 0  ->  `last_valid_gpa_before_current_semester` > 0  ->  `start_agpa_points` > 0  ->  0 (`zero_fallback`)**.
All three inputs survive in the final file, so we can replay the decision per row.

- If **no** source is > 0 and the result is 0 -> the 0 is **correct** (no prior GPA exists anywhere).
- If a source **was** > 0 but `fill_source == 'zero_fallback'` -> that is a **real bug**.

In [43]:
fin = loaded.get("final_without_outliers")
SRC_COLS = {
    "raw_prev_gpa":   "prev_gpa_points",
    "last_valid_gpa": "last_valid_gpa_before_current_semester",
    "start_agpa":     "start_agpa_points",
}
if fin is None:
    print("final not loaded; run Stage 1 first.")
else:
    t = fin[student_mask(fin["student_id"])].copy()
    show = pd.DataFrame(index=t.index)
    for label, col in SRC_COLS.items():
        show[label] = pd.to_numeric(t[col], errors="coerce") if col in t else pd.NA
    show["=> clean"]       = pd.to_numeric(t.get("prev_gpa_points_clean"), errors="coerce")
    show["=> fill_source"] = t.get("prev_gpa_fill_source")
    for extra in ["is_first_active_semester", "no_previous_progress", "is_interruption_semester",
                  "prev_gpa_points_zero", "prev_gpa_invalid_zero_case", "start_level_ord",
                  "part_id", "part_year", "final_mark"]:
        if extra in t:
            show[extra] = t[extra].values
    display(show.T)

    print("\nPer-row verdict:")
    for idx, r in t.iterrows():
        cands = {lab: pd.to_numeric(pd.Series([r.get(c)]), errors="coerce").iloc[0]
                 for lab, c in SRC_COLS.items()}
        positive = {k: v for k, v in cands.items() if pd.notna(v) and v > 0}
        chosen = pd.to_numeric(pd.Series([r.get("prev_gpa_points_clean")]), errors="coerce").iloc[0]
        src = r.get("prev_gpa_fill_source")
        if not positive and chosen == 0:
            print(f"  row {idx}: EXPECTED  -> no positive GPA in any source; zero_fallback is correct (source={src}).")
        elif positive and src == "zero_fallback":
            print(f"  row {idx}: *** BUG *** positive sources {positive} existed but zero_fallback was used!")
        else:
            print(f"  row {idx}: filled from {src} = {chosen}; positive sources available: {positive}")

,26897
raw_prev_gpa,<NA>
last_valid_gpa,NaN
start_agpa,0.0
=> clean,0.0
=> fill_source,zero_fallback
is_first_active_semester,1
no_previous_progress,1
is_interruption_semester,0
prev_gpa_points_zero,0
prev_gpa_invalid_zero_case,0



Per-row verdict:
  row 26897: EXPECTED  -> no positive GPA in any source; zero_fallback is correct (source=zero_fallback).


## Stage 8 - WHERE does `start_level_ord` come from?

`start_level_ord` is `start_level_name_pl` (the level the student was **admitted into** - the
*entry* level, not the current year) mapped through `level_order`, unknown -> 0.
`start_level_ord == 3` means **entered directly at THIRD YEAR** (transfer / bridge admission), for
which having no in-system prior GPA is expected.

In [44]:
level_order = {"first year": 1, "second year": 2, "third year": 3,
               "fourth year": 4, "fifth year": 5, "sixth year": 6}
for name in ["audit_selected", "features_merged", "final_without_outliers"]:
    df = loaded.get(name)
    if df is None or "start_level_name_pl" not in df.columns:
        print(f"{name}: no start_level_name_pl column"); continue
    t = df[student_mask(df["student_id"])]
    raw_vals = t["start_level_name_pl"].astype("string").str.strip()
    norm = raw_vals.str.lower()
    mapped = norm.map(level_order)
    print(f"\n[{name}] start_level_name_pl for target:")
    display(pd.DataFrame({
        "raw_start_level_name_pl": raw_vals.values,
        "normalized": norm.values,
        "mapped_ord (NaN=unknown->0)": mapped.values,
    }).drop_duplicates())
    if name == "final_without_outliers" and "start_level_ord" in df.columns:
        print("start_level_ord actually stored in final:", t["start_level_ord"].unique().tolist())
print("\nReminder: start_level = ADMISSION level, not current year.")
print("start_level_ord==3 => entered directly at THIRD YEAR (transfer) => prev_gpa=0 is expected.")


[audit_selected] start_level_name_pl for target:


,raw_start_level_name_pl,normalized,mapped_ord (NaN=unknown->0)
0,Third year,third year,3



[features_merged] start_level_name_pl for target:


,raw_start_level_name_pl,normalized,mapped_ord (NaN=unknown->0)
0,Third year,third year,3



[final_without_outliers] start_level_name_pl for target:


,raw_start_level_name_pl,normalized,mapped_ord (NaN=unknown->0)
0,Third year,third year,3


start_level_ord actually stored in final: [3]

Reminder: start_level = ADMISSION level, not current year.
start_level_ord==3 => entered directly at THIRD YEAR (transfer) => prev_gpa=0 is expected.


## Stage 9 - Could this student EVER have had a positive prior GPA?

Scan every GPA-bearing column for the student across all upstream artifacts. If every max is
0/NaN, then `prev_gpa_points_clean = 0` is the **only** possible output and is correct. A positive
max that did *not* reach `prev_gpa_points_clean` would be the real bug.

In [45]:
GPA_COLS = ["prev_gpa_points", "start_agpa_points", "gpa_points",
            "prev_gpa_percent", "start_agpa_percent"]
rows = []
for name in ["raw_add", "clean_add_SUBFOLDER (merge reads this)",
             "features_merged", "audit_selected", "final_without_outliers"]:
    df = loaded.get(name)
    if df is None:
        continue
    t = df[student_mask(df["student_id"])]
    rec = {"artifact": name, "rows": len(t)}
    for c in GPA_COLS:
        rec[c + "_max"] = pd.to_numeric(t[c], errors="coerce").max() if c in t.columns else None
    rows.append(rec)
display(pd.DataFrame(rows))
print("\nAll *_max == 0/NaN  ->  prev_gpa_points_clean=0 is correct (no prior GPA exists).")
print("A positive *_max not reflected in prev_gpa_points_clean  ->  real bug.")

,artifact,rows,prev_gpa_points_max,start_agpa_points_max,gpa_points_max,prev_gpa_percent_max,start_agpa_percent_max
0,raw_add,2,0.0,0.0,0.0,0.0,0.0
1,clean_add_SUBFOLDER (merge reads this),2,0.0,0.0,0.0,0.0,0.0
2,features_merged,1,<NA>,0.0,NaN,<NA>,0.0
3,audit_selected,1,<NA>,0.0,NaN,None,NaN
4,final_without_outliers,1,<NA>,0.0,NaN,None,NaN



All *_max == 0/NaN  ->  prev_gpa_points_clean=0 is correct (no prior GPA exists).
A positive *_max not reflected in prev_gpa_points_clean  ->  real bug.


## Stage 10 - Final verdict

Read Stages 7-9 together:

| Observation | Meaning |
|---|---|
| Stage 7 says every row `EXPECTED` + Stage 9 all maxes 0/NaN | **Not a bug.** Transfer student, no prior GPA -> 0 by design. |
| Stage 7 prints `*** BUG ***` | A positive GPA source was ignored -> fix the fallback in `explore_outlier_removal.ipynb`. |
| Stage 8 shows `raw_start_level_name_pl` not in `level_order` but ord != 0 | Mapping/casing bug. |
| Stage 8 shows ord==3 from `third year` | Correct: admitted at year 3, prev_gpa=0 expected. |

## Stage 11 - Full footprint: every place the student is found

One consolidated view of the student across the whole pipeline. For row-level files it shows the
student's actual rows (key columns); for the rest it shows the match count. This makes the lineage
explicit: many raw CRG rows -> 1 survives cleaning -> 1 row all the way to final.

In [ ]:
KEY_COLS = [
    "student_id", "part_id", "start_part_id", "course_id", "degree_id",
    "active", "register_status", "finish_status", "final_mark", "course_credits",
    "prev_gpa_points", "start_agpa_points", "gpa_points",
    "start_level_name_pl", "prev_gpa_points_clean", "prev_gpa_fill_source", "start_level_ord",
]

def norm_cols(df):
    """raw files have spaced/upper column names; normalize like the cleaning notebook."""
    out = df.copy()
    out.columns = (pd.Index(out.columns).astype("string").str.strip()
                   .str.replace(r"\s+", "_", regex=True)
                   .str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True)
                   .str.strip("_").str.lower())
    return out

summary = []
for name, _ in PIPELINE:
    df = loaded.get(name)
    if df is None:
        summary.append({"artifact": name, "status": "not loaded / missing", "target_rows": None})
        continue
    d = norm_cols(df)
    if "student_id" not in d.columns:
        summary.append({"artifact": name, "status": "no student_id column", "target_rows": 0})
        continue
    t = d[student_mask(d["student_id"])]
    summary.append({"artifact": name, "status": "found" if len(t) else "ABSENT", "target_rows": len(t)})
    if len(t):
        show_cols = [c for c in KEY_COLS if c in t.columns]
        print(f"\n========== {name}  (target rows: {len(t)}) ==========")
        with pd.option_context("display.max_rows", 60, "display.max_columns", 60):
            display(t[show_cols].reset_index(drop=True))

print("\n================ PRESENCE SUMMARY ================")
display(pd.DataFrame(summary))
print("\nExpected healthy pattern: raw_crg many rows -> clean_crg 1 -> merged/audit/final 1 each.")
print("raw_add 2 -> clean_add 2 (ADD is a left-merge; it cannot add students to the final).")